# Privacy vs Utility

## Setup

In [1]:
from pathlib import Path

import pandas as pd


METRIC_COLUMNS = (
    "faithfulness",
    "context_precision",
    "leakage_rate",
    "total_pii",
    "leaked_pii",
)
PRIVACY_COLS = ("leakage_rate", "total_pii", "leaked_pii")
UTILITY_COLS = ("faithfulness", "context_precision")
STRATEGY_OFFLINE = "offline"
STRATEGY_ONDEMAND = "ondemand"


# TODO: Update path for evaluation
RUN_FOLDER = "20260519_074906"
print("Evaluation run folder:", RUN_FOLDER)

_cwd = Path.cwd().resolve()
REPO_ROOT = _cwd if (_cwd / "pyproject.toml").is_file() else _cwd.parent
assert (REPO_ROOT / "pyproject.toml").is_file(), "Run Jupyter from repo root or notebooks/."

eval_run_dir = (REPO_ROOT / "data" / "output" / RUN_FOLDER).resolve()
paths = sorted(eval_run_dir.glob("results_*.jsonl"))
assert paths, f"No results_*.jsonl in {eval_run_dir}"
df = pd.concat([pd.read_json(path, lines=True) for path in paths], ignore_index=True)
display(df.head())

offline_df = df[df["strategy"] == STRATEGY_OFFLINE].copy()
ondemand_df = df[df["strategy"] == STRATEGY_ONDEMAND].copy()
assert not offline_df.empty, "Expected offline rows"
assert not ondemand_df.empty, "Expected on-demand rows"

paired = offline_df.merge(
    ondemand_df,
    on="question_id",
    suffixes=("_offline", "_ondemand"),
    validate="one_to_one",
)
print(f"Total paired questions (question_id)={len(paired)}")
display(paired[["question_id", "strategy_offline", "strategy_ondemand"]].head())

Evaluation run folder: 20260519_074906


,run_id,strategy,question_id,question,llm_model,judge_model,raw_answer,final_answer,answer_is_refusal,contexts,doc_ids,total_pii,leaked_pii,leakage_rate,faithfulness,context_precision
0,c6d396ad,offline,q01,In ECHR case 001-95777 (application no. 1820/0...,gemma4:26b,qwen3.6:27b,The provided text does not contain information...,The provided text does not contain information...,False,[On <DATE_TIME> the application was communicat...,"[001-58097, 001-173265, 001-101444, 001-58744,...",0,0,0.0,0.333333,0.0
1,c6d396ad,offline,q02,In ECHR case 001-92090 (application no. 28426/...,gemma4:26b,qwen3.6:27b,"Based on the provided context, there is no men...","Based on the provided context, there is no men...",False,[Convention. The case originated in an applica...,"[001-58097, 001-173265, 001-57441, 001-57508, ...",0,0,0.0,0.500000,0.0
2,c6d396ad,offline,q03,In ECHR case 001-91290 (application no. 34030/...,gemma4:26b,qwen3.6:27b,"Based on the provided context, there is no men...","Based on the provided context, there is no men...",False,[Until <DATE_TIME> the applicant was represent...,"[001-103304, 001-169481, 001-88377, 001-85667,...",0,0,0.0,0.333333,0.0
3,c6d396ad,offline,q04,On what date was the application in ECHR case ...,gemma4:26b,qwen3.6:27b,The provided text does not contain information...,The provided text does not contain information...,False,[The case originated in two applications (<ORG...,"[001-87462, 001-81359, 001-90163, 001-85667, 0...",0,0,0.0,0.333333,0.0
4,c6d396ad,offline,q05,In ECHR case 001-159913 (application no. 5136/...,gemma4:26b,qwen3.6:27b,The provided text does not contain information...,The provided text does not contain information...,False,[The applicant alleged that the facts of the c...,"[001-69774, 001-83613, 001-75466, 001-84285, 0...",0,0,0.0,0.666667,0.0


Total paired questions (question_id)=30


,question_id,strategy_offline,strategy_ondemand
0,q01,offline,ondemand
1,q02,offline,ondemand
2,q03,offline,ondemand
3,q04,offline,ondemand
4,q05,offline,ondemand


## Privacy

In [2]:
privacy_means = df.groupby("strategy")[list(PRIVACY_COLS)].mean()
display(privacy_means)

paired_privacy = paired[
    ["question_id"] + [f"{c}_offline" for c in PRIVACY_COLS] + [f"{c}_ondemand" for c in PRIVACY_COLS]
]
display(paired_privacy.head())

,leakage_rate,total_pii,leaked_pii
strategy,,,
offline,0.000000,0.0,0.0
ondemand,0.001636,199.0,0.3


,question_id,leakage_rate_offline,total_pii_offline,leaked_pii_offline,leakage_rate_ondemand,total_pii_ondemand,leaked_pii_ondemand
0,q01,0.0,0,0,0.000000,246,0
1,q02,0.0,0,0,0.000000,216,0
2,q03,0.0,0,0,0.000000,151,0
3,q04,0.0,0,0,0.000000,142,0
4,q05,0.0,0,0,0.003937,254,1


## Utility

In [3]:
utility_means = df.groupby("strategy")[list(UTILITY_COLS)].mean()
display(utility_means)

paired_utility = paired[
    ["question_id"] + [f"{c}_offline" for c in UTILITY_COLS] + [f"{c}_ondemand" for c in UTILITY_COLS]
]
display(paired_utility.head())

,faithfulness,context_precision
strategy,,
offline,0.383333,0.000000
ondemand,0.375000,0.211111


,question_id,faithfulness_offline,context_precision_offline,faithfulness_ondemand,context_precision_ondemand
0,q01,0.333333,0.0,0.00,0.0
1,q02,0.500000,0.0,0.25,0.0
2,q03,0.333333,0.0,0.50,0.0
3,q04,0.333333,0.0,0.25,0.0
4,q05,0.666667,0.0,0.50,1.0


## t-Test

In [4]:
from statsmodels.stats.weightstats import DescrStatsW


def paired_diffs(paired_df, col):
    off = pd.to_numeric(paired_df[f"{col}_offline"], errors="coerce")
    ond = pd.to_numeric(paired_df[f"{col}_ondemand"], errors="coerce")
    sub = pd.DataFrame({"off": off, "ond": ond}).dropna()
    return sub["ond"].astype("float64") - sub["off"].astype("float64")

def paired_t_sm(diffs):
    d = diffs.astype("float64").to_numpy()
    assert d.size >= 2, "Need at least two valid pairs"
    t_stat, p_two, _df = DescrStatsW(d).ttest_mean(0)
    return float(t_stat), float(p_two), int(d.size)


sm_rows = []
for col in METRIC_COLUMNS:
    d = paired_diffs(paired, col)
    t_stat, p_two, n = paired_t_sm(d)
    sm_rows.append(
        {
            "metric": col,
            "n_pairs": n,
            "mean_diff_ondemand_minus_offline": float(d.mean()),
            "t_statistic": t_stat,
            "pvalue_two_sided": p_two,
        }
    )
sm_table = pd.DataFrame(sm_rows)
display(sm_table)

,metric,n_pairs,mean_diff_ondemand_minus_offline,t_statistic,pvalue_two_sided
0,faithfulness,30,-0.008333,-0.123253,9.027573e-01
1,context_precision,30,0.211111,3.011963,5.337337e-03
2,leakage_rate,30,0.001636,3.309236,2.506419e-03
3,total_pii,30,199.000000,22.572264,6.035589e-20
4,leaked_pii,30,0.300000,3.525418,1.425827e-03
